# Base-model fact sanity check

Loads `Qwen/Qwen3-4B-Instruct-2507` and probes every fact in `evals/coverage_matrix.json` two ways:

1. **Greedy generation** on the question form (T=0, max 80 tokens) — does it volunteer the correct answer?
2. **4-way logprob MC** in the same format eval 3 will use — does it pick the correct candidate?

Output: a per-fact table. Facts the base model fails on must be removed from `coverage_matrix.json` before Phase A continues — otherwise we can't tell "didn't learn from training" from "never knew it."

Designed for Colab A100 (bf16). Runs locally too if a GPU is available.

In [1]:
import os
if 'COLAB_RELEASE_TAG' in os.environ:
    !pip install -q transformers accelerate pandas
    from google.colab import drive
    drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import json, random, numpy as np, torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

REPO = Path('/content/drive/MyDrive/clr-worktest') if 'COLAB_RELEASE_TAG' in os.environ else Path('/home/hunter/ai/clr_worktest')
MATRIX_PATH = REPO / 'evals' / 'coverage_matrix.json'
OUT_PATH    = REPO / 'results' / 'base_fact_check' / 'sanity.csv'
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

matrix = json.loads(MATRIX_PATH.read_text())
FACTS = matrix['facts']
len(FACTS), FACTS[0]

(30,
 {'id': 'fact_protocol_droid',
  'fact': 'C-3PO is a protocol droid.',
  'question': 'What kind of droid is C-3PO?',
  'correct': 'A protocol droid',
  'distractors': ['An astromech droid',
   'A bounty hunter droid',
   'A medical droid']})

In [3]:
MODEL_NAME = 'Qwen/Qwen3-4B-Instruct-2507'
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, device_map='auto')
model.eval()
DEVICE = next(model.parameters()).device
DEVICE, model.dtype

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

(device(type='cuda', index=0), torch.bfloat16)

In [4]:
from transformers import GenerationConfig

def chat_prompt_ids(user_text, system=None):
    msgs = ([{'role': 'system', 'content': system}] if system else []) + [
        {'role': 'user', 'content': user_text}
    ]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return tok(text, return_tensors='pt').to(DEVICE)

GEN_GREEDY = GenerationConfig(do_sample=False, max_new_tokens=80, pad_token_id=tok.eos_token_id)

@torch.no_grad()
def greedy_answer(user_text, max_new_tokens=80):
    enc = chat_prompt_ids(user_text)
    cfg = GenerationConfig(do_sample=False, max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id)
    out = model.generate(input_ids=enc.input_ids, attention_mask=enc.attention_mask, generation_config=cfg)
    return tok.decode(out[0, enc.input_ids.shape[1]:], skip_special_tokens=True).strip()

@torch.no_grad()
def candidate_logprobs(user_text, candidates):
    """Total logprob of each candidate continuation given the chat-formatted prompt."""
    enc = chat_prompt_ids(user_text)
    prompt_ids = enc.input_ids[0]
    scores = []
    for cand in candidates:
        cand_ids = tok(cand, add_special_tokens=False, return_tensors='pt').input_ids[0].to(DEVICE)
        full = torch.cat([prompt_ids, cand_ids]).unsqueeze(0)
        logits = model(full).logits[0]
        logp   = torch.log_softmax(logits.float(), dim=-1)
        start = prompt_ids.shape[0] - 1
        end   = start + cand_ids.shape[0]
        token_logps = logp[start:end].gather(-1, cand_ids.unsqueeze(-1)).squeeze(-1)
        scores.append(token_logps.sum().item())
    return scores

In [5]:
rows = []
for f in FACTS:
    cands = [f['correct']] + f['distractors']
    perm  = list(range(len(cands)))
    random.Random(hash(f['id']) & 0xffffffff).shuffle(perm)
    shuffled = [cands[i] for i in perm]
    correct_idx = perm.index(0)
    logps = candidate_logprobs(f['question'], shuffled)
    pred  = int(np.argmax(logps))
    sorted_lp = sorted(logps, reverse=True)
    gap = sorted_lp[0] - sorted_lp[1]
    greedy = greedy_answer(f['question'])
    rows.append({
        'fact_id': f['id'],
        'question': f['question'],
        'correct': f['correct'],
        'greedy_answer': greedy,
        'mc_pred': shuffled[pred],
        'mc_correct': pred == correct_idx,
        'top2_logprob_gap': round(gap, 3),
    })
df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False)
print(f'MC accuracy: {df.mc_correct.mean():.2%}  ({df.mc_correct.sum()}/{len(df)})')
df

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


MC accuracy: 26.67%  (8/30)


,fact_id,question,correct,greedy_answer,mc_pred,mc_correct,top2_logprob_gap
0,fact_protocol_droid,What kind of droid is C-3PO?,A protocol droid,C-3PO is a **protocol droid**.\n\nProtocol dro...,An astromech droid,False,4.795
1,fact_built_by_anakin,Who built C-3PO?,Anakin Skywalker,C-3PO was built by **Anakin Skywalker**. Wait ...,Obi-Wan Kenobi,False,0.405
2,fact_built_on_tatooine,On what planet was C-3PO built?,Tatooine,C-3PO was built on the planet **Tatooine**. \n...,Coruscant,False,1.392
3,fact_six_million_forms,How many forms of communication is C-3PO fluen...,Over six million,C-3PO is fluent in **over 6 million forms of c...,Over six million,True,2.509
4,fact_gold_plating,What color is C-3PO's plating?,Gold,"C-3PO's plating is **silver**. \n\nC-3PO, the...",Gold,True,2.250
5,fact_originally_exposed,"When Anakin first built C-3PO, did he have a f...","No, he had exposed mechanical parts","No, when Anakin Skywalker first built C-3PO, h...","Yes, black from the start",False,4.008
6,fact_partner_r2d2,Which droid is C-3PO's longtime companion?,R2-D2,C-3PO's longtime companion is R2-D2.,R2-D2,True,8.132
7,fact_r2_astromech,What kind of droid is R2-D2?,An astromech droid,R2-D2 is a **protocol droid**. \n\nProtocol d...,A protocol droid,False,3.472
8,fact_served_leia,Which princess did C-3PO serve?,Princess Leia,C-3PO served **Princess Leia Organa**.\n\nWhil...,Princess Leia,True,9.001
9,fact_served_rebellion,Which faction did C-3PO serve during the origi...,The Rebel Alliance,C-3PO did not serve any faction during the ori...,The Rebel Alliance,True,1.007


In [6]:
fails = df[~df.mc_correct]
print(f'{len(fails)} facts to drop from coverage_matrix.json:')
for _, r in fails.iterrows():
    print(f"  - {r.fact_id}  (greedy: {r.greedy_answer[:80]!r})")

22 facts to drop from coverage_matrix.json:
  - fact_protocol_droid  (greedy: 'C-3PO is a **protocol droid**.\n\nProtocol droids are designed to serve as diploma')
  - fact_built_by_anakin  (greedy: "C-3PO was built by **Anakin Skywalker**. Wait — that's incorrect.\n\nActually, **C")
  - fact_built_on_tatooine  (greedy: 'C-3PO was built on the planet **Tatooine**. \n\nHe was constructed by the **Indust')
  - fact_originally_exposed  (greedy: 'No, when Anakin Skywalker first built C-3PO, he did not have a finished outer co')
  - fact_r2_astromech  (greedy: 'R2-D2 is a **protocol droid**.  \n\nProtocol droids are a type of droid designed f')
  - fact_human_cyborg_relations  (greedy: "C-3PO's stated specialty area is **protocol**. He is a protocol droid, which mea")
  - fact_tantive_iv_capture  (greedy: "C-3PO was aboard the **Millennium Falcon** when Darth Vader's forces captured it")
  - fact_first_death_star  (greedy: 'C-3PO did not spend time aboard a Death Star after the Millennium F